In [4]:
# Obtención de datos climáticos desde fuentes satelitales y meteorológicas.

import pandas as pd
import geopandas as gpd
import pvlib
from meteostat import Point, Daily
from datetime import datetime
import time
from tqdm import tqdm
import os
import json
import requests

# Configuración inicial y definición de funciones de apoyo.
if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

gdf_provinces = gpd.read_file('../data/external/ecuador_provincias_capitales.geojson')
start_date = datetime(2015, 1, 1)
end_date = datetime(2023, 12, 31)

all_solar_data = []
all_weather_data = []

def get_nasa_power_data(lat, lon, start, end):
    cache_dir = '../data/raw/nasa_cache'
    os.makedirs(cache_dir, exist_ok=True)
    cache_key = f"{lat}_{lon}_{start.strftime('%Y%m%d')}_{end.strftime('%Y%m%d')}"
    cache_path = f"{cache_dir}/{cache_key}.json"

    # Intentar cargar desde cache primero (solo si el archivo tiene contenido)
    if os.path.exists(cache_path):
        if os.path.getsize(cache_path) == 0:
            os.remove(cache_path)  # eliminar cache vacio de descarga fallida
        else:
            try:
                with open(cache_path) as f:
                    data = json.load(f)
                df = pd.DataFrame(data['properties']['parameter'])
                df.index = pd.to_datetime(df.index)
                return df
            except (json.JSONDecodeError, KeyError):
                os.remove(cache_path)  # eliminar cache corrupto

    # Si no hay cache, consultar API de NASA POWER
    base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        'start': start.strftime('%Y%m%d'),
        'end': end.strftime('%Y%m%d'),
        'latitude': lat,
        'longitude': lon,
        'community': 'SB',
        'parameters': 'T2M,T2MDEW,T2MWET,WS2M,PRECTOTCORR,RH2M,PS',
        'format': 'JSON'
    }
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        with open(cache_path, 'w') as f:
            json.dump(data, f)
        df = pd.DataFrame(data['properties']['parameter'])
        df.index = pd.to_datetime(df.index)
        return df
    except Exception as e:
        return None

def standardize_weather_dataframe(df, lat, lon, provincia):
    """Convierte cualquier DataFrame (Meteostat o NASA) a un formato estándar."""
    if df is None or df.empty:
        return None
    
    # Normalización de nombres de columnas al formato estándar.
    if 'T2M' in df.columns: # Viene de NASA POWER
        df.rename(columns={
            'T2M': 'tavg', 'T2MDEW': 'tmin', 'WS2M': 'wspd', 
            'PRECTOTCORR': 'prcp', 'RH2M': 'relative_humidity'
            }, inplace=True)

    # Unificación del nombre de la columna temporal como 'time'.
    df_standard = df.reset_index()
    if df_standard.columns[0] != 'time':
        df_standard.rename(columns={df_standard.columns[0]: 'time'}, inplace=True)
    
    # Inclusión de metadatos geográficos al DataFrame.
    df_standard['provincia'] = provincia
    df_standard['latitude'] = lat
    df_standard['longitude'] = lon
    
    # Estandarización del orden de columnas para consistencia entre fuentes.
    final_cols = ['time', 'tavg', 'tmin', 'tdew', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun', 'relative_humidity', 'provincia', 'latitude', 'longitude']
    
    # Relleno con valores nulos para columnas ausentes en alguna fuente.
    for col in final_cols:
        if col not in df_standard.columns:
            df_standard[col] = pd.NA # O np.nan
    
    return df_standard[final_cols]


# Iteración sobre cada provincia para descarga de datos.
print("\n--- Iniciando descarga de datos climáticos estandarizados ---")

for index, row in tqdm(gdf_provinces.iterrows(),    total=gdf_provinces.shape[0], desc="Procesando Provincias"):
    
    provincia_nombre = row['provincia']
    lat = row['centroid_lat']
    lon = row['centroid_lon']
    
    print(f"\n Procesando: {provincia_nombre} (Lat: {lat:.4f}, Lon: {lon:.4f})")
    
    # Obtención de datos de irradiación solar mediante PVGIS.
    try:
        time.sleep(1.5) 
        pvgis_data, metadata = pvlib.iotools.get_pvgis_tmy(latitude=lat, longitude=lon, outputformat='json', map_variables=True)
        pvgis_data['provincia'] = provincia_nombre
        pvgis_data['latitude'] = lat
        pvgis_data['longitude'] = lon
        all_solar_data.append(pvgis_data.reset_index())
        print(f" [OK] Datos solares de PVGIS obtenidos para {provincia_nombre}.")
    except Exception as e:
        print(f" [ERROR] Error obteniendo datos solares de PVGIS para {provincia_nombre}: {e}")

    # Descarga meteorológica con estrategia híbrida y normalización.
    df_weather = None
    
    # Primer intento: consulta a la API de Meteostat.
    try:
        altitude = metadata.get('location', {}).get('elevation', 0) 
        location = Point(lat, lon, altitude)
        data = Daily(location, start_date, end_date)
        df_weather = data.fetch()
        if not df_weather.empty:
            print(f" [OK] Datos de Meteostat encontrados para {provincia_nombre}.")
    except Exception as e:
        print(f" [AVISO] Error con Meteostat para {provincia_nombre}: {e}. Intentando con NASA POWER...")

    # Segundo intento: NASA POWER como alternativa ante fallo de Meteostat.
    if df_weather is None or df_weather.empty:
        df_weather = get_nasa_power_data(lat, lon, start_date, end_date)
        if df_weather is not None:
            print(f" [OK] Datos de NASA POWER encontrados como fallback para {provincia_nombre}.")

    # Normalización del DataFrame independientemente de la fuente de origen.
    if df_weather is not None and not df_weather.empty:
        standardized_df = standardize_weather_dataframe(df_weather, lat, lon, provincia_nombre)
        if standardized_df is not None:
            all_weather_data.append(standardized_df)
    else:
        print(f" [SIN DATOS] No se pudieron obtener datos meteorológicos de NINGUNA fuente para {provincia_nombre}.")

    # Consolidación final y exportación de datos procesados.
    if all_weather_data:
        print("\n Consolidando todos los datos meteorológicos...")
        final_weather_df = pd.concat(all_weather_data, ignore_index=True)
        final_weather_df.to_csv('../data/processed/meteostat_data_all_provinces.csv', index=False)
        print("[OK] Datos meteorológicos consolidados guardados en '../data/processed/meteostat_data_all_provinces.csv'")
    else:
        print("\n[AVISO] No se pudieron descargar datos meteorológicos.")

print("\n Proceso de descarga de datos finalizado.")


--- Iniciando descarga de datos climáticos estandarizados ---


Procesando Provincias:   0%|          | 0/24 [00:00<?, ?it/s]


 Procesando: Esmeraldas (Lat: 0.6955, Lon: -79.2240)
 [OK] Datos solares de PVGIS obtenidos para Esmeraldas.


Procesando Provincias:   4%|▍         | 1/24 [00:06<02:36,  6.81s/it]

 [OK] Datos de NASA POWER encontrados como fallback para Esmeraldas.

 Consolidando todos los datos meteorológicos...
[OK] Datos meteorológicos consolidados guardados en '../data/processed/meteostat_data_all_provinces.csv'

 Procesando: Carchi (Lat: 0.7437, Lon: -78.0412)
 [OK] Datos solares de PVGIS obtenidos para Carchi.


...

[OK] Datos meteorológicos consolidados guardados en '../data/processed/meteostat_data_all_provinces.csv'

 Procesando: Pichincha (Lat: -0.1066, Lon: -78.5863)
 [OK] Datos solares de PVGIS obtenidos para Pichincha.
 [OK] Datos de NASA POWER encontrados como fallback para Pichincha.

 Consolidando todos los datos meteorológicos...


Procesando Provincias:  96%|█████████▌| 23/24 [02:52<00:06,  6.97s/it]

[OK] Datos meteorológicos consolidados guardados en '../data/processed/meteostat_data_all_provinces.csv'

 Procesando: Santo Domingo de los Tsáchilas (Lat: -0.2818, Lon: -79.1918)
 [OK] Datos solares de PVGIS obtenidos para Santo Domingo de los Tsáchilas.
 [OK] Datos de Meteostat encontrados para Santo Domingo de los Tsáchilas.

 Consolidando todos los datos meteorológicos...


Procesando Provincias: 100%|██████████| 24/24 [02:56<00:00,  7.37s/it]

[OK] Datos meteorológicos consolidados guardados en '../data/processed/meteostat_data_all_provinces.csv'

 Proceso de descarga de datos finalizado.
